# SpectraShift Week 2: BigEarthNet staging and split freeze
Attach only the source snapshot without `data/sealed/`. With Internet enabled, the notebook streams the two revision-pinned TorchGeo mirror parts directly, checks their exact published sizes, and does not store the 63 GB source archive in `/kaggle/working`.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/spectrashift-week2')
WORK.mkdir(parents=True, exist_ok=True)
archives = sorted(INPUT.rglob('BigEarthNet-S2.tar.zst'))
mirror_parts = sorted([*INPUT.rglob('BigEarthNet-S2.tar.gzaa'), *INPUT.rglob('BigEarthNet-S2.tar.gzab')])
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift').is_dir()]
if not projects:
    source_bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(source_bundles) == 1, f'Expected one SpectraShift source bundle, found {source_bundles}'
    extracted_source = WORK / 'source'
    shutil.unpack_archive(str(source_bundles[0]), str(extracted_source))
    projects = [extracted_source]
assert len(archives) <= 1, f'Expected at most one official archive, found {archives}'
assert not mirror_parts or [p.name for p in mirror_parts] == ['BigEarthNet-S2.tar.gzaa', 'BigEarthNet-S2.tar.gzab'], f'Attach both mirror parts or neither; found {mirror_parts}'
assert len(projects) == 1, f'Expected one SpectraShift source tree, found {projects}'
ARCHIVE, ARCHIVE_PARTS, PROJECT = (archives[0] if archives else None), mirror_parts, projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
print({'project': str(PROJECT), 'archive': str(ARCHIVE) if ARCHIVE else None, 'archive_parts': [str(p) for p in ARCHIVE_PARTS], 'working': str(WORK)})

In [ ]:
config = yaml.safe_load((PROJECT / 'configs/data/week2.yaml').read_text())
if ARCHIVE:
    config['dataset']['archive_path'] = str(ARCHIVE)
    config['dataset'].pop('archive_parts', None)
else:
    config['dataset'].pop('archive_path', None)
    config['dataset'].pop('archive_md5', None)
    config['dataset']['archive_parts'] = [str(path) for path in ARCHIVE_PARTS] if ARCHIVE_PARTS else list(config['dataset']['archive_mirror_urls'].values())
config['dataset']['candidate_manifest'] = str(PROJECT / 'manifests/draft/candidates.parquet')
config['dataset']['sealed_candidate_labels'] = '/sealed-labels-are-not-mounted'
config['staging']['output_dir'] = str(WORK / 'staged')
config['staging']['final_manifest_dir'] = str(WORK / 'manifest-v1')
config['staging']['sealed_output_dir'] = '/sealed-output-is-not-mounted'
config['staging']['report_dir'] = str(WORK / 'reports')
runtime_config = WORK / 'week2-kaggle.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print(runtime_config.read_text())

In [ ]:
from spectrashift.data.preflight import run_preflight
preflight = run_preflight(runtime_config)
print(json.dumps(preflight, indent=2))
assert preflight['ready'], 'Kaggle working storage or archive input is insufficient'
import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    device_arch = f'sm_{major}{minor}'
    supported_arches = torch.cuda.get_arch_list()
    print({'gpu': torch.cuda.get_device_name(0), 'device_arch': device_arch, 'pytorch_arches': supported_arches})
    assert not supported_arches or device_arch in supported_arches, f'{torch.cuda.get_device_name(0)} ({device_arch}) is unsupported by this PyTorch build; select a T4 GPU'

In [ ]:
from spectrashift.data.archive import stage_archive
staging = stage_archive(runtime_config)
print(json.dumps(staging, indent=2))

In [ ]:
from spectrashift.data.freeze import freeze_split
from spectrashift.data.normalization import compute_normalization
freeze = freeze_split(runtime_config)
assert freeze['training_approved'], freeze
normalization = compute_normalization(runtime_config)
print(json.dumps({'freeze': freeze, 'normalization_sha256': normalization['sha256']}, indent=2))

In [ ]:
from spectrashift.train.throughput import benchmark_throughput
throughput = benchmark_throughput(runtime_config)
print(json.dumps(throughput, indent=2))

In [ ]:
from spectrashift.data.smoke import run_smoke
smoke = run_smoke(runtime_config)
assert smoke['overfit_gate'], smoke
(WORK / 'week2_run_summary.json').write_text(json.dumps({'preflight': preflight, 'staging': staging, 'freeze': freeze, 'normalization_sha256': normalization['sha256'], 'throughput': throughput, 'smoke': smoke}, indent=2) + '\n')
print(json.dumps(smoke, indent=2))